I create this because I want to have a draft to implement the forecasting workflow in Continuous Ensemble Weather Forecasting.

I can use claude of course, but I cannot ensure I can express what I mean 100%, so code a draft and use claude then.

Actually I am not 100% precent sure the way I should implement this, so it's kind of an exploration as well. 

In [1]:
import numpy as np
import csv
import sys
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from natsort import natsorted

from diffusion_networks import SongUNet
from dataset import  DATA_100, SQGLeadTimeDataset
from loss import flow_matching_loss
from cond_sampler import CondSampler
from utils import compute_rmse, compute_crps, compute_ssr, compute_metrics, plot_metrics


ROOT = Path('..').resolve()          # repo root
sys.path.insert(0, str(ROOT / 'Condition_model'))

# ── shared config ────────────────────────────────────────────────────────────
DATA_DIR     = DATA_100
DATA_STD     = 2660.0
MODEL_PATH   = ROOT / 'models' / 'randomAnchor_n_24pred_test.pth'
batch_size   = 1
MAX_HORIZON  = 99          # matches training
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'device : {device}')
print(f'data   : {DATA_DIR}')
print(f'model : {MODEL_PATH}')

device : cuda
data   : /home/trace/Documents/Projects/Thesis/data
model : /home/trace/Documents/Projects/Thesis/models/randomAnchor_n_24pred_test.pth


new dataset file

include 3 modification:

modify: 

the initial could be other frame except the first one, so the current frame has to be the later one, and previous frame has to be the earlier one

the temporal resolution of training data could be bigger than 1

in the eval mode, the dataset could return a batch consisting of different lead time samples


In [2]:
# test code for dataset
# in the val mode, can this perform get a batch with different lead time samples?
# n_init is a good tool to control the samples size, but just need to rename it, terrile name
test_train_ds = SQGLeadTimeDataset(split='train', random_lead_time=True)


test_loader = DataLoader(test_train_ds, batch_size=25, shuffle=True)
# the shuffle = false is default

x0, xt, t = next(iter(test_loader))
print(x0.shape)
print(xt.shape)
print(t.shape)
print(t)

torch.Size([25, 2, 64, 64])
torch.Size([25, 2, 64, 64])
torch.Size([25])
tensor([1.0000, 1.0000, 0.2917, 0.5000, 0.4583, 0.7083, 0.0417, 0.8750, 0.6667,
        0.2500, 0.6250, 0.5417, 0.6250, 0.5417, 0.7083, 0.3333, 0.1250, 0.6667,
        0.5833, 0.7083, 0.5833, 0.7917, 0.0417, 0.4167, 0.2083])


In [3]:
new_ds = SQGLeadTimeDataset(split='val', random_lead_time=False, eval_traj_num=50)
# what if i use data loader?
# Create DataLoader
loader = DataLoader(new_ds, batch_size=1, shuffle=True)

initial, target, lead_time , traj_idx, frame_idx= next(iter(loader))

print(lead_time.shape)
print(initial.shape)
print(target.shape)

print(lead_time)
print(f"It is the {frame_idx}th frame in the trajectory, it can support {(100-frame_idx) / 24} times AR.")


torch.Size([1, 24])
torch.Size([1, 2, 64, 64])
torch.Size([1, 24, 2, 64, 64])
tensor([[0.0417, 0.0833, 0.1250, 0.1667, 0.2083, 0.2500, 0.2917, 0.3333, 0.3750,
         0.4167, 0.4583, 0.5000, 0.5417, 0.5833, 0.6250, 0.6667, 0.7083, 0.7500,
         0.7917, 0.8333, 0.8750, 0.9167, 0.9583, 1.0000]])
It is the tensor([35])th frame in the trajectory, it can support tensor([2.7083]) times AR.


In [2]:
"""
Conditional Flow-Matching Training.

Trains p(x_{t+k} | x_t, lead=k) via linear stochastic interpolant.

Key changes from original:
- Dataset now samples anchor t from any valid frame (not just frame 0).
- max_lead=24, leads k in {0,1,...,24}; k=0 included for embedding coverage.
- LABEL_DROPOUT=0.0 (CFG not used in this project).
- FILTERS=32, DATA_100 for fast iteration; swap to DATA_500/FILTERS=64 for real runs.
"""


# ============================================================================
# Config
# ============================================================================

# ── data ──────────────────────────────────────────────────────────────────
DATA_DIR       = DATA_100    # swap to DATA_500 for full training run
DATA_STD       = 2660.0
MAX_LEAD       = 24          # direct leads: k in {0, 1, ..., 24}
MAX_FRAMES     = 100         # fixed by sqg_generate.py

# ── model ─────────────────────────────────────────────────────────────────
IMG_CHANNELS   = 2
IMG_RESOLUTION = 64
FILTERS        = 32          # small for quick sanity runs; use 64 for real training
LABEL_DROPOUT  = 0.0         # CFG not used — must be 0.0

# ── training ──────────────────────────────────────────────────────────────
BATCH_SIZE   = 16
NUM_EPOCHS   = 300
LR           = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_ITERS = 500

# ── misc ──────────────────────────────────────────────────────────────────
SAVE_PATH = ROOT / 'models' / 'randomAnchor_n_24pred_test.pth'
LOG_PATH  = ROOT / 'models' / 'randomAnchor_n_24pred_test_training_log.csv'
PLOT_PATH = ROOT / 'models' / 'randomAnchor_n_24pred_test_loss_curves.png'


# ============================================================================
# Builders
# ============================================================================

def build_datasets():
    """
    Build train/val datasets.

    Training mode returns (initial, target, time_label):
        initial    : (C, H, W)  — anchor frame traj[t], t drawn from any valid position
        target     : (C, H, W)  — traj[t + k], k in {0, 1, ..., MAX_LEAD}
        time_label : scalar     — k / MAX_LEAD in [0, 1]

    k=0 pairs (target == initial, label=0.0) are included to anchor the
    time embedding at zero; they contribute near-zero loss.
    """
    common = dict(
        std            = DATA_STD,
        max_lead       = MAX_LEAD,
        max_frames     = MAX_FRAMES,
        random_lead_time = True,    # training mode
        size            = 0.05
    )

    train_dataset = SQGLeadTimeDataset(DATA_DIR, split='train', **common)
    val_dataset   = SQGLeadTimeDataset(DATA_DIR, split='val',   **common)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True,
    )

    print(f'Leads     : {train_dataset.forecasting_leads}')
    print(f'Pairs     — train: {len(train_dataset):,}, val: {len(val_dataset):,}')
    return train_loader, val_loader


def build_model(device):
    """
    SongUNet with in_channels=4: [z_s (2ch)] cat [x_t (2ch)].
    time_emb=1 enables the lead-time Fourier embedding (map_time).
    label_dropout=0.0: no CFG dropout.
    """
    model = SongUNet(
        img_resolution     = IMG_RESOLUTION,
        in_channels        = IMG_CHANNELS * 2,  # 2 (z_s) + 2 (x_t conditioning)
        out_channels       = IMG_CHANNELS,
        embedding_type     = 'fourier',
        encoder_type       = 'residual',
        decoder_type       = 'standard',
        channel_mult_noise = 2,
        resample_filter    = [1, 3, 3, 1],
        model_channels     = FILTERS,
        channel_mult       = [2, 2, 2],
        attn_resolutions   = [32],
        label_dropout      = LABEL_DROPOUT,     # 0.0 — no CFG
        time_emb           = 1,                 # must be 1 for lead-time conditioning
    ).to(device)

    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
    return model


# ============================================================================
# Training
# ============================================================================

def run_epoch(model, loader, device, optimizer=None, warmup=None, desc=''):
    """One epoch. Trains when optimizer is given, otherwise evaluates."""
    train = optimizer is not None
    model.train() if train else model.eval()

    total = 0.0
    torch.set_grad_enabled(train)
    iterator = tqdm(loader, desc=desc, leave=False) if train else loader

    for initial, target, lead_time in iterator:
        # initial  : (B, C, H, W) — conditioning anchor
        # target   : (B, C, H, W) — generation target
        # lead_time: (B,)         — normalised lead in [0, 1]
        initial   = initial.to(device)
        target    = target.to(device)
        lead_time = lead_time.to(device)

        if train:
            optimizer.zero_grad()
            loss = flow_matching_loss(model, initial, target, lead_time)
            loss.backward()
            optimizer.step()
            warmup.step()
        else:
            loss = flow_matching_loss(model, initial, target, lead_time)

        total += loss.item()

    torch.set_grad_enabled(True)
    return total / len(loader)


def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device : {device}')
    print(f'Data   : {DATA_DIR}')

    SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader = build_datasets()
    model = build_model(device)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    warmup    = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_ITERS
    )

    train_losses, val_losses = [], []
    best_val_loss  = float('inf')
    patience       = 20
    patience_counter = 0

    with open(LOG_PATH, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'val_loss'])

    for epoch in range(NUM_EPOCHS):
        avg_train = run_epoch(
            model, train_loader, device, optimizer, warmup,
            desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]',
        )
        avg_val = run_epoch(model, val_loader, device)

        train_losses.append(avg_train)
        val_losses.append(avg_val)
        scheduler.step()

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), SAVE_PATH)
            tag = '  <- best'
            patience_counter = 0
        else:
            tag = ''
            patience_counter += 1

        with open(LOG_PATH, 'a', newline='') as f:
            csv.writer(f).writerow([epoch + 1, avg_train, avg_val])
        print(f'Epoch {epoch+1:3d}  train={avg_train:.4f}  val={avg_val:.4f}{tag}')

        if patience_counter >= patience:
            print(f'\nEarly stopping at epoch {epoch+1}.')
            break

    print(f'\nTraining done. Best val loss: {best_val_loss:.4f}')
    plot_losses(train_losses, val_losses)
    return train_losses, val_losses


# ============================================================================
# Loss curves
# ============================================================================

def plot_losses(train_losses, val_losses):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='train')
    plt.plot(val_losses,   label='val')
    plt.xlabel('Epoch')
    plt.ylabel('Flow-matching loss')
    plt.title(f'p(x_{{t+k}} | x_t, lead=k)  —  max_lead={MAX_LEAD}, filters={FILTERS}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=150)
    print(f'Loss curves saved to {PLOT_PATH}')


if __name__ == '__main__':
    train()

Device : cuda
Data   : /home/trace/Documents/Projects/Thesis/data
Leads     : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Pairs     — train: 7,600, val: 182,400
Parameters: 3,543,778


KeyboardInterrupt: 

In [3]:
sampler = CondSampler(model_path=MODEL_PATH, device=device, steps=100, eps=None)

In [4]:
from forecasting import forecast

results = forecast(sampler, eval_traj_num=3)

100%|██████████| 3/3 [05:32<00:00, 110.78s/it]


In [4]:
print(results[0][0].shape)
print(results[1][0].shape)
print(results[2][0].shape)
print(len(results))

torch.Size([2, 24, 5, 2, 64, 64])
torch.Size([2, 24, 5, 2, 64, 64])
torch.Size([1, 24, 5, 2, 64, 64])
3


In [5]:
from utils import visualize_results

visualize_results(results)

Saved → ../Visual/sample00.gif
Saved → ../Visual/sample01.gif
Saved → ../Visual/sample02.gif


it's time to design a metrics compute function 

In [ ]:
max_frames = max(p.shape[0] * p.shape[1] for p, t in results)
n_ens = results[0][0].shape[2]
rmse_sum   = np.zeros(max_frames)
spread_sum = np.zeros(max_frames)
crps_sum   = np.zeros(max_frames)
counts     = np.zeros(max_frames)

for predicts, truth in results:
    n_steps, n_lead = predicts.shape[:2]
    T = n_steps * n_lead
    p = predicts.reshape(T, *predicts.shape[2:]).numpy()
    t = truth.reshape(T, *truth.shape[2:]).numpy()

    rmse_sum[:T]   += compute_rmse(p.mean(axis=1), t)
    spread_sum[:T] += p.std(axis=1, ddof=1).mean(axis=(1,2,3))
    crps_sum[:T]   += compute_crps(p, t)
    counts[:T]     += 1

rmse_mean   = rmse_sum / counts
spread_mean = spread_sum / counts
crps_mean   = crps_sum / counts
ssr = np.sqrt((n_ens+1)/n_ens) * spread_mean / (rmse_mean + 1e-8)


lead_times = np.arange(1, max_frames + 1)
valid = counts > 0   # only plot positions with at least one sample

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(lead_times[valid], rmse_mean[valid], color='steelblue')
axes[0].set_title('RMSE'); axes[0].set_xlabel('Lead time (h)'); axes[0].grid(alpha=0.3)

axes[1].plot(lead_times[valid], crps_mean[valid], color='darkorange')
axes[1].set_title('CRPS'); axes[1].set_xlabel('Lead time (h)'); axes[1].grid(alpha=0.3)

axes[2].plot(lead_times[valid], ssr[valid], color='seagreen')
axes[2].axhline(1.0, color='red', linestyle='--', lw=1, label='ideal')
axes[2].set_title('SSR'); axes[2].set_xlabel('Lead time (h)'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'Metrics over lead time — {int(counts[0])} samples', fontsize=11)
plt.tight_layout()
plt.savefig('../Performance/metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [5]:
metrics = compute_metrics(results)
plot_metrics(metrics)

/home/trace/Documents/Projects/Thesis/Condition_model/utils.py:221: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
